# IBRD World Bank Loans: Data Ingestion & Exploration

This notebook loads the raw IBRD loan snapshot and performs a non-destructive exploratory audit of schema, missingness, duplicates, and value distributions before any cleaning.

In [18]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pio.renderers.default = 'notebook'

candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
project_root = next(
    (root for root in candidate_roots if (root / 'data' / 'raw' / 'ibrd_statement_of_loans_and_guarantees_latest_available_snapshot_09-09-2026.csv').exists()),
    Path.cwd().resolve(),
)
raw_path = project_root / 'data' / 'raw' / 'ibrd_statement_of_loans_and_guarantees_latest_available_snapshot_09-09-2026.csv'
print(f'Project root: {project_root}')
print(f'Raw path: {raw_path}')


Project root: /home/rigii/ATA
Raw path: /home/rigii/ATA/data/raw/ibrd_statement_of_loans_and_guarantees_latest_available_snapshot_09-09-2026.csv


In [19]:
if not raw_path.exists():
    print(f'Raw data file not found at: {raw_path}')
    print(f'Looking for files in: {raw_path.parent}')
    available_files = list(raw_path.parent.glob('*.csv'))
    if available_files:
        print(f'Available CSV files: {[f.name for f in available_files]}')
        raw_path = available_files[0]
        print(f'Using: {raw_path}')
    else:
        raise FileNotFoundError(f'No CSV files found in {raw_path.parent}. Please ensure the data file exists at {raw_path}')

df = pd.read_csv(raw_path, encoding='utf-8-sig', low_memory=False)

if df.columns[0].startswith('\ufeff'):
    df.columns = [col.replace('\ufeff', '') for col in df.columns]

print(f'Raw shape: {df.shape}')
print('\nFirst 10 rows:')
display(df.head(10))
print('\nDataFrame info:')
df.info()
print('\nDescriptive statistics:')
display(df.describe(include='all').T)


Raw shape: (9518, 35)

First 10 rows:


,End of Period,Loan Number,Region,Country / Economy Code,Country / Economy,Borrower,Guarantor Country / Economy Code,Guarantor,Loan Type,Loan Status,Interest Rate,Currency of Commitment,Project ID,Project Name,Original Principal Amount (US$),Cancelled Amount (US$),Undisbursed Amount (US$),Disbursed Amount (US$),Repaid to IBRD (US$),Due to IBRD (US$),Exchange Adjustment (US$),Borrower's Obligation (US$),Sold 3rd Party (US$),Repaid 3rd Party (US$),Due 3rd Party (US$),Loans Held (US$),First Repayment Date,Last Repayment Date,Agreement Signing Date,Board Approval Date,Effective Date (Most Recent),Closed Date (Most Recent),Last Disbursement Date,Board approval - Fiscal year,Board approval - Calendar year
0,08/31/2026,IBRD87030,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.0,NaN,P153173,Anhui Road Maintenance Innovation,150000000.0,1.209400e+07,0.0,1.379060e+08,2.364696e+07,1.142590e+08,0.00,1.142590e+08,0.0,0.0,0,1.142590e+08,05/15/2023,11/15/2036,04/11/2017,02/21/2017,08/17/2017,10/31/2025,04/07/2026,2017,2017
1,08/31/2026,IBRD87040,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.0,NaN,P153604,Poyang Lake Water Environment Management,150000000.0,1.410283e+07,0.0,1.358972e+08,6.237680e+06,1.296595e+08,0.00,1.296595e+08,0.0,0.0,0,1.296595e+08,09/15/2025,03/15/2042,06/06/2017,03/16/2017,10/13/2017,12/31/2022,05/24/2023,2017,2017
2,08/31/2026,IBRD87200,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.0,NaN,P154623,China: Gansu TVET Project,120000000.0,1.419267e+06,0.0,1.185807e+08,1.640274e+07,1.021780e+08,0.00,1.021780e+08,0.0,0.0,0,1.021780e+08,03/01/2023,03/01/2047,06/26/2017,03/31/2017,10/20/2017,06/30/2023,12/13/2023,2017,2017
3,08/31/2026,IBRD87440,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.0,NaN,P154984,China Health Reform Program,600000000.0,1.914958e+06,0.0,5.980064e+08,8.233229e+07,5.156741e+08,1511927.12,5.171860e+08,0.0,0.0,0,5.156741e+08,10/01/2022,04/01/2051,06/30/2017,05/09/2017,09/11/2017,12/31/2021,08/16/2022,2017,2017
4,08/31/2026,IBRD87660,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.0,NaN,P153473,Three Gorges Modern Logistics Center,200000000.0,1.109539e+07,0.0,1.889046e+08,1.356343e+07,1.753412e+08,0.00,1.753412e+08,0.0,0.0,0,1.753412e+08,11/01/2023,05/01/2047,09/01/2017,06/30/2017,12/27/2017,12/31/2024,08/21/2025,2017,2017
5,08/31/2026,IBRD87770,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.0,NaN,P153115,Hunan Integr. Manag of Agri Land Project,100000000.0,2.692622e+06,0.0,9.730738e+07,1.432991e+07,8.297747e+07,0.00,8.297747e+07,0.0,0.0,0,8.297747e+07,12/01/2023,06/01/2043,12/11/2017,08/22/2017,03/07/2018,12/31/2023,06/13/2024,2018,2017
6,08/31/2026,IBRD87910,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.0,NaN,P154621,China: Guangdong Compulsory Education,120000000.0,2.593846e+06,0.0,1.174062e+08,1.285632e+07,1.045498e+08,0.00,1.045498e+08,0.0,0.0,0,1.045498e+08,05/01/2023,11/01/2042,01/16/2018,10/31/2017,04/02/2018,11/30/2023,05/28/2024,2018,2017
7,08/31/2026,IBRD88000,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.0,NaN,P147009,Jiangxi Farm Produce Distribution System,150000000.0,3.422599e+06,0.0,1.465774e+08,2.375402e+07,1.228234e+08,0.00,1.228234e+08,0.0,0.0,0,1.228234e+08,06/01/2023,12/01/2041,03/05/2018,12/15/2017,05/15/2018,06/30/2025,12/11/2025,2018,2017
8,08/31/2026,IBRD88370,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.0,NaN,P158713,Liaoning Safe and Sustainable Urban WS,250000000.0,1.034868e+08,0.0,1.465132e+08,1.474794e+07,1.317653e+08,0.00,1.317653e+08,0.0,0.0,0,1.317653e+08,11/01/2024,05/01/2044,08/06/2018,06/06/2018,10/23/2018,06/30/2024,12/05/2024,2018,2018
9,08/31/2026,IBRD88460,EAST ASIA AND PACIFIC,CN,China,MINISTRY OF FINANCE,CN,China,FSL,Repaying,0.0,NaN,P158717,China: Hubei Inland Waterway Improvement,150000000.0,5.474629e+04,0.0,1.4994


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9518 entries, 0 to 9517
Data columns (total 35 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   End of Period                     9518 non-null   object 
 1   Loan Number                       9518 non-null   object 
 2   Region                            9518 non-null   object 
 3   Country / Economy Code            9515 non-null   object 
 4   Country / Economy                 9518 non-null   object 
 5   Borrower                          9461 non-null   object 
 6   Guarantor Country / Economy Code  9235 non-null   object 
 7   Guarantor                         9238 non-null   object 
 8   Loan Type                         9518 non-null   object 
 9   Loan Status                       9518 non-null   object 
 10  Interest Rate                     9409 non-null   float64
 11  Currency of Commitment            0 non-null      fl

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
End of Period,9518,1,08/31/2026,9518,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Loan Number,9518,9518,IBRD87030,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Region,9518,7,LATIN AMERICA AND CARIBBEAN,2975,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country / Economy Code,9515,147,ID,657,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country / Economy,9518,148,Indonesia,657,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Borrower,9461,973,MINISTRY OF FINANCE,1285,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Guarantor Country / Economy Code,9235,126,ID,657,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Guarantor,9238,127,Indonesia,657,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Loan Type,9518,11,FSL,2916,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Loan Status,9518,11,Fully Repaid,6622,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1) Data Quality Audit

This section checks missingness, uniqueness, and duplicate identifiers to find data quality risks before analysis.

In [20]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing_count / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct
}).reset_index().rename(columns={'index': 'column'})

print('Missing values by column:')
display(missing_summary.head(20))

print('\nColumns with >50% missing:')
display(missing_summary[missing_summary['missing_pct'] > 50])

print('\nUnique values per column:')
display(df.nunique(dropna=True).sort_values(ascending=False).head(20).to_frame(name='unique_count'))

loan_number_dupes = df['Loan Number'].duplicated(keep=False)
print(f'\nDuplicate loan number rows: {loan_number_dupes.sum()}')
if loan_number_dupes.any():
    dup_rows = df.loc[loan_number_dupes, ['Loan Number', 'Region', 'Country / Economy', 'Loan Status', 'Board Approval Date']].copy()
    dup_rows = dup_rows.sort_values(['Loan Number', 'Board Approval Date'])
    display(dup_rows.head(20))


Missing values by column:


,column,missing_count,missing_pct
0,Currency of Commitment,9518,100.00
1,Last Disbursement Date,3587,37.69
2,Guarantor Country / Economy Code,283,2.97
3,Guarantor,280,2.94
4,Interest Rate,109,1.15
5,First Repayment Date,61,0.64
6,Last Repayment Date,60,0.63
7,Borrower,57,0.60
8,Project Name,26,0.27
9,Closed Date (Most Recent),8,0.08



Columns with >50% missing:


,column,missing_count,missing_pct
0,Currency of Commitment,9518,100.0



Unique values per column:


,unique_count
Loan Number,9518
Project ID,7295
Repaid to IBRD (US$),7061
Disbursed Amount (US$),7006
Project Name,6270
Effective Date (Most Recent),5776
Agreement Signing Date,5132
Cancelled Amount (US$),4558
Last Disbursement Date,4025
Board Approval Date,3638



Duplicate loan number rows: 0


## 2) Data Type Analysis

This section calls out fields that look numeric or datetime in concept but are still stored as strings in the raw export.

In [21]:
numeric_like_cols = [
    'Original Principal Amount (US$)', 'Cancelled Amount (US$)', 'Undisbursed Amount (US$)',
    'Disbursed Amount (US$)', 'Repaid to IBRD (US$)', 'Due to IBRD (US$)',
    'Exchange Adjustment (US$)', "Borrower's Obligation (US$)", 'Loans Held (US$)',
    'Interest Rate', 'Sold 3rd Party (US$)', 'Repaid 3rd Party (US$)', 'Due 3rd Party (US$)',
    'Board approval - Fiscal year', 'Board approval - Calendar year'
]

numeric_candidates = []
for col in numeric_like_cols:
    if col not in df.columns:
        continue
    non_null = df[col].dropna().astype(str)
    if len(non_null) == 0:
        continue
    numeric_mask = pd.to_numeric(non_null.str.replace(',', '', regex=False), errors='coerce').notna()
    if df[col].dtype == 'object' and not numeric_mask.all():
        numeric_candidates.append(col)

date_like_cols = [
    'End of Period', 'First Repayment Date', 'Last Repayment Date',
    'Agreement Signing Date', 'Board Approval Date',
    'Effective Date (Most Recent)', 'Closed Date (Most Recent)',
    'Last Disbursement Date'
]

date_candidates = []
for col in date_like_cols:
    if col not in df.columns:
        continue
    non_null = df[col].dropna().astype(str)
    if len(non_null) == 0:
        continue
    parsed = pd.to_datetime(non_null, errors='coerce')
    if (parsed.notna()).sum() > 0 and (parsed.notna()).sum() < len(non_null):
        date_candidates.append(col)

type_summary = pd.DataFrame({
    'column': numeric_candidates + date_candidates,
    'type_group': ['numeric_string'] * len(numeric_candidates) + ['datetime_string'] * len(date_candidates)
})

print('Columns that should be numeric but are strings:')
display(pd.DataFrame({'column': numeric_candidates}))

print('\nColumns that should be datetime but are strings:')
display(pd.DataFrame({'column': date_candidates}))

print('\nSummary table:')
display(type_summary)


Columns that should be numeric but are strings:


,column



Columns that should be datetime but are strings:


,column



Summary table:


,column,type_group


## 3) Basic Distributions

This section profiles the most important categorical dimensions and loan-size distribution for the raw portfolio.

In [22]:
for col in ['Region', 'Loan Status', 'Loan Type', 'Currency of Commitment']:
    if col in df.columns:
        print(f'\nValue counts for {col}:')
        display(df[col].fillna('Missing').astype(str).value_counts().head(15).to_frame(name='count'))

if 'Country / Economy' in df.columns:
    print('\nTop 10 countries by number of loans:')
    display(df['Country / Economy'].fillna('Missing').astype(str).value_counts().head(10).to_frame(name='loan_count'))

if 'Original Principal Amount (US$)' in df.columns:
    original_amount = pd.to_numeric(df['Original Principal Amount (US$)'], errors='coerce')
    print('\nSummary stats for Original Principal Amount (US$):')
    display(original_amount.describe().to_frame(name='value'))



Value counts for Region:


,count
Region,
LATIN AMERICA AND CARIBBEAN,2975
EAST ASIA AND PACIFIC,2014
EUROPE AND CENTRAL ASIA,1948
"MID EAST,NORTH AFRICA,AFG,PAK",1256
SOUTH ASIA,466
EASTERN AND SOUTHERN AFRICA,438
WESTERN AND CENTRAL AFRICA,421



Value counts for Loan Status:


,count
Loan Status,
Fully Repaid,6622
Repaying,1207
Disbursing,531
Fully Disbursed,266
Fully Cancelled,220
Fully Transferred,210
Disbursing&Repaying,203
Terminated,79
Approved,71



Value counts for Loan Type:


...

,count
Currency of Commitment,
Missing,9518



Top 10 countries by number of loans:


,loan_count
Country / Economy,
Indonesia,657
Brazil,530
China,497
India,439
Mexico,384
Turkiye,372
Morocco,328
Philippines,310
Argentina,295



Summary stats for Original Principal Amount (US$):


,value
count,9.518000e+03
mean,1.046016e+08
std,1.803428e+08
min,0.000000e+00
25%,1.430367e+07
50%,4.000000e+07
75%,1.147850e+08
max,3.750000e+09


## 4) Initial Observations

Key issues to note before any cleaning or modeling.

In [23]:
numeric_cols = [
    'Original Principal Amount (US$)', 'Cancelled Amount (US$)', 'Undisbursed Amount (US$)',
    'Disbursed Amount (US$)', 'Repaid to IBRD (US$)', 'Due to IBRD (US$)',
    'Exchange Adjustment (US$)', "Borrower's Obligation (US$)", 'Loans Held (US$)'
]

summary = {
    'columns_over_50pct_missing': missing_summary[missing_summary['missing_pct'] > 50]['column'].tolist(),
    'zero_original_amount_loans': int((pd.to_numeric(df['Original Principal Amount (US$)'], errors='coerce') == 0).sum()),
    'negative_amount_columns': [
        col for col in numeric_cols
        if col in df.columns and (pd.to_numeric(df[col], errors='coerce') < 0).any()
    ],
    'date_formats_seen': sorted({str(v)[:10] for v in df['Board Approval Date'].dropna().head(10)})
}

print('Summary of raw data issues:')
for key, value in summary.items():
    print(f'{key}: {value}')


Summary of raw data issues:
columns_over_50pct_missing: ['Currency of Commitment']
zero_original_amount_loans: 314
negative_amount_columns: ['Undisbursed Amount (US$)', 'Due to IBRD (US$)', 'Exchange Adjustment (US$)', "Borrower's Obligation (US$)", 'Loans Held (US$)']
date_formats_seen: ['02/21/2017', '03/16/2017', '03/31/2017', '05/09/2017', '05/18/2018', '06/06/2018', '06/30/2017', '08/22/2017', '10/31/2017', '12/15/2017']


- Columns with more than 50% missing values should be reviewed carefully before downstream modeling because they may have limited analytical value.
- The raw dataset includes loans with an original amount of zero, which should be checked to distinguish true zero-amount records from data-entry anomalies.
- Some monetary fields may include negative values, which can signal adjustments, reversals, or data anomalies and should be interpreted cautiously.
- Date fields are stored in string form using Excel-style US date formats such as `MM/DD/YYYY`, and must be converted to datetime objects before time-based analysis.
- The portfolio appears to include many historical loans and a large number of loan statuses, so duplicate loan numbers and date ambiguity should be considered before building time series or repayment models.